# 01 · Main models: propensity score matching (PSM) and double machine learning (DML)
Associations between each greenery measure and 2016 prescribing per registered patient, for wards and, as a check at a finer scale, LSOAs (Tables 1 and 2). The estimators are in `causal.py`.

- **PSM (Table 1):** treatment = greenery above the citywide median; 1-nearest-neighbour matching on the logit of the propensity score (caliper 0.2 SD); average treatment effect (ATE) as % of the below-median mean; `N_BOOT` bootstrap resamples give the standard error and the 95 % interval.
- **DML (Table 2):** greenery kept continuous; effect of one SD more greenery from a cross-fitted partially linear model with random forests (5 folds × 5 repetitions), as % of mean prescribing.

**Inputs:** `data/data_analysis_wards7s.csv`, `data/data_analysis_lsoas7s.csv`.
**Outputs:** `results/models/psm_{wards,lsoas}.csv`, `results/models/dml_{wards,lsoas}.csv`.
**Runtime:** PSM about 5 minutes per level; DML about 20 minutes for wards and over an hour for LSOAs (8 cores). Run from this folder.

In [1]:
from pathlib import Path

import pandas as pd

from causal import CONFOUNDERS, OUTCOMES, TREATMENTS, rename_columns, run_grid, run_grid_continuous, table1

REPO = Path.cwd().resolve().parents[1]
OUT_DIR = REPO / 'results' / 'models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT = 1000
ALPHA = 0.05
SEED = 0
data = {lv: rename_columns(pd.read_csv(REPO / 'data' / f'data_analysis_{lv}7s.csv')) for lv in ('wards', 'lsoas')}
print({lv: len(d) for lv, d in data.items()})

{'wards': 632, 'lsoas': 4835}


## 1. PSM → SAVE

In [2]:
psm = {}
for lv, d in data.items():
    psm[lv] = run_grid(d, treatments=TREATMENTS, outcomes=OUTCOMES, confounders=CONFOUNDERS,
                       n_boot=N_BOOT, seed=SEED, alpha=ALPHA, verbose=False)
    psm[lv].to_csv(OUT_DIR / f'psm_{lv}.csv', index=False)
    print(lv)
    display(table1(psm[lv], value='ATE_pct', err='SE_pct'))

wards


Outcome,diabetes,hypertension,asthma,opioids,total
Label,,,,,
NDVI,-2.32±3.17,-0.14±1.71,-0.81±1.63,2.33±3.48,0.49±1.98
WHO,0.81±2.62,-1.78±1.65,0.14±1.41,4.21±2.58,-1.39±1.68
ESA WHO,0.97±2.67,0.24±1.28,-0.61±1.49,1.15±3.15,-0.05±1.74
NE,0.51±2.63,0.28±1.30,-1.00±1.33,0.12±2.61,0.13±1.76
"on-road, aerial",-2.93±2.06,-1.59±1.84,-1.20±1.58,1.77±3.07,1.14±1.73
"on-road, aerial, weighted",**-4.89±2.11**,-3.34±2.25,-2.28±1.85,3.03±3.21,-1.40±1.86
"on-road, street view",**-4.24±2.40**,-1.98±2.57,-1.02±2.02,1.55±3.79,-0.93±2.16
"on-road, street view, weighted",**-5.00±2.15**,-2.42±2.57,-2.68±2.11,0.45±3.67,-0.93±2.07
off-road,-3.77±2.23,-2.60±1.50,-2.52±1.35,-1.46±2.47,-3.48±1.64


lsoas


Outcome,diabetes,hypertension,asthma,opioids,total
Label,,,,,
NDVI,-1.14±1.01,-0.33±.65,0.12±.60,-0.63±1.22,**2.15±.73**
WHO,-0.96±1.00,-0.31±.65,-0.24±.57,1.52±.99,-0.97±.70
ESA WHO,-0.45±.97,-0.40±.56,0.08±.61,0.91±1.15,0.49±.72
NE,2.26±.99,0.56±.55,0.43±.53,1.86±1.05,0.55±.73
"on-road, aerial",**-5.60±.85**,**-3.68±.67**,**-2.51±.55**,-0.50±1.04,**-1.79±.74**
"on-road, aerial, weighted",**-5.07±.85**,**-3.11±.73**,**-2.22±.61**,-1.23±1.05,**-1.85±.75**
"on-road, street view",**-6.32±.83**,**-3.39±.81**,**-2.81±.68**,-1.95±1.15,**-2.61±.82**
"on-road, street view, weighted",**-6.74±.80**,**-2.85±.77**,**-1.97±.66**,-0.11±1.22,**-1.89±.83**
off-road,-1.05±.91,-0.97±.64,-0.16±.58,2.27±1.08,0.27±.69


## 2. Double ML → SAVE

In [3]:
dml = {}
for lv, d in data.items():
    dml[lv] = run_grid_continuous(d, treatments=TREATMENTS, outcomes=OUTCOMES, confounders=CONFOUNDERS,
                                  learner='forest', seed=SEED, alpha=ALPHA, verbose=False)
    dml[lv].to_csv(OUT_DIR / f'dml_{lv}.csv', index=False)
    print(lv)
    display(table1(dml[lv], value='Effect_pct', err='SE_pct'))

wards


Outcome,diabetes,hypertension,asthma,opioids,total
Label,,,,,
NDVI,-0.14±.74,-0.22±.55,0.08±.69,1.23±1.24,1.38±.71
WHO,-0.87±.67,-0.71±.39,0.00±.53,1.24±1.02,**-1.22±.62**
ESA WHO,0.04±.69,0.10±.51,0.43±.61,0.77±1.07,-0.13±.65
NE,-0.58±.65,-0.34±.45,-0.11±.51,1.16±1.05,-0.70±.66
"on-road, aerial",-0.70±.69,-0.54±.53,0.15±.59,1.07±1.13,0.55±.67
"on-road, aerial, weighted",-1.03±.71,-0.89±.50,-0.26±.58,1.10±1.13,-0.22±.69
"on-road, street view",**-2.14±.69**,-0.73±.56,0.00±.69,1.31±1.26,-0.59±.71
"on-road, street view, weighted",**-2.44±.72**,-0.70±.57,0.01±.70,1.47±1.32,-0.81±.76
off-road,**-1.34±.59**,-0.96±.54,-0.39±.49,-0.64±.92,-0.18±.64


lsoas


Outcome,diabetes,hypertension,asthma,opioids,total
Label,,,,,
NDVI,-0.01±.31,0.09±.24,-0.20±.24,0.09±.48,**0.68±.30**
WHO,0.23±.27,0.05±.18,0.19±.22,**1.02±.40**,-0.22±.30
ESA WHO,-0.29±.33,-0.11±.22,0.35±.23,**1.01±.43**,-0.05±.26
NE,**0.86±.27**,**0.46±.19**,0.33±.22,**0.96±.40**,0.24±.30
"on-road, aerial",**-1.66±.30**,**-1.20±.22**,**-0.89±.23**,-0.74±.45,-0.49±.32
"on-road, aerial, weighted",**-1.86±.30**,**-1.20±.23**,**-0.88±.23**,-0.81±.46,**-0.75±.34**
"on-road, street view",**-2.95±.30**,**-0.85±.22**,**-0.72±.24**,**-1.12±.47**,**-1.16±.32**
"on-road, street view, weighted",**-2.73±.29**,**-0.71±.22**,**-0.48±.23**,-0.76±.47,**-0.97±.33**
off-road,-0.30±.27,**-0.49±.22**,-0.16±.21,-0.18±.40,0.02±.31


## Validation
PSM should balance the controls (max |standardised mean difference| after matching below 0.1); the DML `Treatment_R2` shows how much of each measure the controls already explain.

In [4]:
for lv in data:
    print(lv, '| unbalanced PSM cells:', int((~psm[lv]['Balanced']).sum()),
          '| DML treatment R2 by measure:', dml[lv].groupby('Label', sort=False)['Treatment_R2'].mean().round(2).to_dict())

wards | unbalanced PSM cells: 10 | DML treatment R2 by measure: {'NDVI': 0.41, 'WHO': 0.01, 'ESA WHO': 0.21, 'NE': 0.11, 'on-road, aerial': 0.28, 'on-road, aerial, weighted': 0.3, 'on-road, street view': 0.38, 'on-road, street view, weighted': 0.41, 'off-road': 0.01}
lsoas | unbalanced PSM cells: 0 | DML treatment R2 by measure: {'NDVI': 0.29, 'WHO': 0.0, 'ESA WHO': 0.11, 'NE': 0.03, 'on-road, aerial': 0.22, 'on-road, aerial, weighted': 0.23, 'on-road, street view': 0.22, 'on-road, street view, weighted': 0.21, 'off-road': -0.03}
